# 🩺 تدريب الذكاء الاصطناعي الحقيقي (Kaggle RANZCR CLiP)
هذا الملف سيقوم بتحميل 40,000 صورة أشعة حقيقية من موقع Kaggle وتدريب النظام عليها.
## الخطوات التحضيرية:
1. اذهب إلى موقع [Kaggle.com](https://www.kaggle.com) وقم بتسجيل الدخول.
2. اذهب إلى إعدادات حسابك (Settings) واضغط على `Create New API Token`. سيتم تحميل ملف باسم `kaggle.json`.
3. قم بتشغيل الخلية التالية لرفع ملف `kaggle.json` الخاص بك.

## 🔑 ربط حساب Kaggle تلقائياً
قم بتشغيل هذه الخلية، سيظهر لك زر **Upload (رفع)**. اضغط عليه واختر ملف `kaggle.json` الذي قمت بتحميله. سيقوم الكود بوضعه في المكان الصحيح تلقائياً دون الحاجة لكتابة أي شيء.

In [ ]:
from google.colab import files
import os

print('يرجى رفع ملف kaggle.json الذي قمت بتحميله من حسابك:')
uploaded = files.upload()

if 'kaggle.json' in uploaded:
    !mkdir -p ~/.kaggle
    !mv kaggle.json ~/.kaggle/
    !chmod 600 ~/.kaggle/kaggle.json
    print('✅ تم ربط الحساب وتجهيز المفتاح بنجاح!')
else:
    print('❌ لم تقم برفع الملف، يرجى المحاولة مرة أخرى.')

## 📥 تحميل وتجهيز البيانات الحقيقية (حوالي 12 جيجابايت)

In [ ]:
!pip install kaggle
!kaggle competitions download -c ranzcr-clip-catheter-line-classification
!unzip -q ranzcr-clip-catheter-line-classification.zip -d ranzcr_data
print('تم تحميل وفك ضغط البيانات الحقيقية بنجاح!')

In [ ]:
import cv2
import torch
import numpy as np
import pandas as pd
import ast
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using device:', DEVICE)

In [ ]:
# -- بنية Attention U-Net --
class ConvBlock(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, 3, padding=1), nn.BatchNorm2d(out_channels), nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, 3, padding=1), nn.BatchNorm2d(out_channels), nn.ReLU(inplace=True)
        )
    def forward(self, x): return self.conv(x)

class AttentionBlock(nn.Module):
    def __init__(self, F_g, F_l, F_int):
        super().__init__()
        self.W_g = nn.Sequential(nn.Conv2d(F_g, F_int, kernel_size=1, stride=1, padding=0, bias=True), nn.BatchNorm2d(F_int))
        self.W_x = nn.Sequential(nn.Conv2d(F_l, F_int, kernel_size=1, stride=1, padding=0, bias=True), nn.BatchNorm2d(F_int))
        self.psi = nn.Sequential(nn.Conv2d(F_int, 1, kernel_size=1, stride=1, padding=0, bias=True), nn.BatchNorm2d(1), nn.Sigmoid())
        self.relu = nn.ReLU(inplace=True)
    def forward(self, g, x):
        g1 = self.W_g(g)
        x1 = self.W_x(x)
        psi = self.relu(g1 + x1)
        psi = self.psi(psi)
        return x * psi

class UpConv(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.up = nn.Sequential(nn.Upsample(scale_factor=2), nn.Conv2d(in_channels, out_channels, kernel_size=3, stride=1, padding=1, bias=True), nn.BatchNorm2d(out_channels), nn.ReLU(inplace=True))
    def forward(self, x): return self.up(x)

class UNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.Maxpool = nn.MaxPool2d(kernel_size=2, stride=2)
        self.Conv1 = ConvBlock(1, 64)
        self.Conv2 = ConvBlock(64, 128)
        self.Conv3 = ConvBlock(128, 256)
        self.Up3 = UpConv(256, 128)
        self.Att3 = AttentionBlock(F_g=128, F_l=128, F_int=64)
        self.Up_conv3 = ConvBlock(256, 128)
        self.Up2 = UpConv(128, 64)
        self.Att2 = AttentionBlock(F_g=64, F_l=64, F_int=32)
        self.Up_conv2 = ConvBlock(128, 64)
        self.Conv_1x1 = nn.Conv2d(64, 1, kernel_size=1, stride=1, padding=0)
    def forward(self, x):
        e1 = self.Conv1(x)
        e2 = self.Maxpool(e1)
        e2 = self.Conv2(e2)
        e3 = self.Maxpool(e2)
        e3 = self.Conv3(e3)
        d3 = self.Up3(e3)
        x2 = self.Att3(g=d3, x=e2)
        d3 = torch.cat((x2, d3), dim=1)
        d3 = self.Up_conv3(d3)
        d2 = self.Up2(d3)
        x1 = self.Att2(g=d2, x=e1)
        d2 = torch.cat((x1, d2), dim=1)
        d2 = self.Up_conv2(d2)
        out = self.Conv_1x1(d2)
        return torch.sigmoid(out)

class DiceLoss(nn.Module):
    def forward(self, inputs, targets):
        inputs, targets = inputs.view(-1), targets.view(-1)
        intersection = (inputs * targets).sum()
        return 1 - (2.*intersection + 1.0)/(inputs.sum() + targets.sum() + 1.0)


In [ ]:
# -- تجهيز البيانات (مواءمة الإحداثيات لرسم الأقنعة) --
class RANZCRDataset(Dataset):
    def __init__(self, csv_file, img_dir):
        self.df = pd.read_csv(csv_file)
        # الفلترة لتسريع التدريب كبداية (يمكنك إزالة هذا السطر لتدريب 40 ألف صورة كلها)
        self.df = self.df.head(5000)
        self.img_dir = img_dir
    def __len__(self):
        return len(self.df)
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img_path = os.path.join(self.img_dir, row['StudyInstanceUID'] + '.jpg')
        
        img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
        if img is None: # في حالة الصورة مفقودة
            return torch.zeros((1, 512, 512)), torch.zeros((1, 512, 512))
            
        mask_orig = np.zeros(img.shape, dtype=np.uint8)
        coords = ast.literal_eval(row['data'])
        pts = np.array(coords, np.int32).reshape((-1, 1, 2))
        cv2.polylines(mask_orig, [pts], isClosed=False, color=255, thickness=20)
        
        img_resized = cv2.resize(img, (512, 512))
        mask_resized = cv2.resize(mask_orig, (512, 512))
        
        return torch.tensor(img_resized, dtype=torch.float32).unsqueeze(0)/255.0, torch.tensor(mask_resized, dtype=torch.float32).unsqueeze(0)/255.0

print('تجهيز البيانات...')
dataset = RANZCRDataset('ranzcr_data/train_annotations.csv', 'ranzcr_data/train')
loader = DataLoader(dataset, batch_size=4, shuffle=True)
print('جاهز للتدريب! عدد الصور:', len(dataset))

In [ ]:
# -- التدريب الحقيقي --
model = UNet().to(DEVICE)
optimizer = optim.Adam(model.parameters(), lr=1e-4)
criterion = DiceLoss()

EPOCHS = 10
for epoch in range(EPOCHS):
    model.train()
    total_loss = 0
    for i, (imgs, masks) in enumerate(loader):
        if imgs.sum() == 0: continue # تجاهل الصور التالفة
        imgs, masks = imgs.to(DEVICE), masks.to(DEVICE)
        optimizer.zero_grad()
        loss = criterion(model(imgs), masks)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
        if i % 100 == 0:
            print(f'Batch {i}/{len(loader)} Loss: {loss.item():.4f}')
    print(f'=== Epoch {epoch+1} Completed | Average Loss: {total_loss/len(loader):.4f} ===')

torch.save(model.state_dict(), 'best_unet_model.pth')
print('✅ انتهى التدريب الحقيقي! قم بتحميل best_unet_model.pth من اليسار.')